## **Aim**
To implement a program that extracts URLs, email addresses, IP addresses, and domain names from a digital evidence text file.

## **Algorithm**
**Step 1:** Import `re`, `collections`, and `urllib.parse` libraries.

**Step 2:** Create a sample digital evidence text file containing various artifacts.

**Step 3:** Define regex patterns for:
   - URLs (http/https/ftp)
   - Email addresses
   - IPv4 addresses
   - IPv6 addresses (basic)
   - Domain names

**Step 4:** Read the evidence file and apply regex patterns.

**Step 5:** Deduplicate and categorize findings.

**Step 6:** Generate a structured report with counts and unique values.

In [1]:
import re
from collections import Counter
from urllib.parse import urlparse

URL_REGEX = re.compile(
    r'(?:https?|ftp)://'
    r'(?:[\w-]+\.)+[\w-]+'
    r'(?:/[^\s]*)?'
    r'(?:\?[^\s]*)?'
    r'(?:#[^\s]*)?'
)

EMAIL_REGEX = re.compile(
    r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
)

IPV4_REGEX = re.compile(
    r'\b(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}'
    r'(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\b'
)

IPV6_REGEX = re.compile(
    r'\b(?:[0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}\b'
)

DOMAIN_REGEX = re.compile(
    r'\b(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b'
)

def create_sample_evidence(filepath):
    content = """
Digital Evidence Case File - CASE-2026-001
==========================================

Suspicious network activity detected from IP 192.168.1.10
Connection to malicious C2 server at 10.0.0.100 on port 443
Lateral movement from 172.16.0.50 to internal network

Phishing URLs found in user browser history:
- https://www.google.com/search?q=forensics
- https://github.com/user/repo
- http://malicious-site.tk/payload.exe
- https://paypal-login.verify-account.ml/login
- https://bit.ly/3xYzK9
- https://amazon.com.phishing-site.ga/update
- https://secure-bank-login.cf/verify
- ftp://files.example.com/evidence.zip
- https://docs.google.com/document/d/123/edit
- https://www.linkedin.com/feed/

Email communications:
- From: admin@company.com
- To: attacker@malicious.com
- CC: support@paypal.com
- Security alerts from: security@microsoft.com
- Automated: noreply@github.com
- Abuse contact: abuse@iana.org

Suspicious IP addresses in logs:
- 192.168.1.10 (internal server)
- 10.0.0.100 (C2 server)
- 172.16.0.50 (attacker)
- 203.0.113.45 (Tor exit node)
- 198.51.100.23 (VPN/Proxy)
- 8.8.8.8 (Google DNS)
- 1.1.1.1 (Cloudflare DNS)
- 192.168.100.50 (C2)

Domain indicators:
- google.com (legitimate)
- github.com (legitimate)
- malicious-site.tk (malicious)
- paypal-login.verify-account.ml (phishing)
- bit.ly (URL shortener)
- amazon.com.phishing-site.ga (phishing)
- secure-bank-login.cf (phishing)
- files.example.com (file hosting)
- docs.google.com (legitimate)
- linkedin.com (legitimate)
- microsoft.com (legitimate)
- example.com (reserved)

File hashes:
- MD5: d41d8cd98f00b204e9800998ecf8427e
- SHA256: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855

Registry artifacts:
- HKCU\Software\Microsoft\Windows\CurrentVersion\Run
- HKLM\Software\Microsoft\Windows\CurrentVersion\Run

Timestamp: 2026-08-20T14:30:00Z
Analyst: Forensic Team Alpha
"""
    with open(filepath, "w") as f:
        f.write(content)

def extract_iocs(text):
    urls = URL_REGEX.findall(text)
    emails = EMAIL_REGEX.findall(text)
    ipv4s = IPV4_REGEX.findall(text)
    ipv6s = IPV6_REGEX.findall(text)
    
    # Extract domains from URLs and standalone
    domains = set()
    for url in urls:
        try:
            parsed = urlparse(url)
            if parsed.netloc:
                domains.add(parsed.netloc.lower())
        except:
            pass
    
    # Also find standalone domains
    for match in DOMAIN_REGEX.findall(text):
        # Filter out IPs and common false positives
        if not IPV4_REGEX.match(match) and not match.endswith(('.exe', '.dll', '.sys', '.zip', '.pdf', '.doc', '.txt')):
            domains.add(match.lower())
    
    ips = ipv4s + ipv6s
    
    return {
        "urls": urls,
        "emails": emails,
        "ips": ips,
        "domains": list(domains)
    }

def main():
    filepath = "evidence_text.txt"
    create_sample_evidence(filepath)
    
    with open(filepath, "r") as f:
        text = f.read()
    
    print("Extracting IOCs from digital evidence file...")
    iocs = extract_iocs(text)
    
    print(f"\n{'='*60}")
    print(f"DIGITAL EVIDENCE IOC EXTRACTION REPORT")
    print(f"{'='*60}")
    print(f"Source file: {filepath}")
    print(f"File size: {len(text)} bytes")
    
    # URLs
    unique_urls = list(set(iocs["urls"]))
    print(f"\n--- URLS ({len(iocs['urls'])} found, {len(unique_urls)} unique) ---")
    for url in sorted(unique_urls):
        print(url)
    
    # Emails
    unique_emails = list(set(iocs["emails"]))
    print(f"\n--- EMAIL ADDRESSES ({len(iocs['emails'])} found, {len(unique_emails)} unique) ---")
    for email in sorted(unique_emails):
        print(email)
    
    # IPs
    unique_ips = list(set(iocs["ips"]))
    print(f"\n--- IP ADDRESSES ({len(iocs['ips'])} found, {len(unique_ips)} unique) ---")
    for ip in sorted(unique_ips):
        print(ip)
    
    # Domains
    unique_domains = sorted(iocs["domains"])
    print(f"\n--- DOMAIN NAMES ({len(iocs['domains'])} found, {len(unique_domains)} unique) ---")
    for domain in unique_domains:
        print(domain)
    
    # Summary
    total = len(iocs["urls"]) + len(iocs["emails"]) + len(iocs["ips"]) + len(iocs["domains"])
    unique_total = len(unique_urls) + len(unique_emails) + len(unique_ips) + len(unique_domains)
    print(f"\n--- SUMMARY ---")
    print(f"Total IOCs extracted: {total}")
    print(f"Unique IOCs: {unique_total}")
    print(f"  URLs: {len(unique_urls)}")
    print(f"  Emails: {len(unique_emails)}")
    print(f"  IPs: {len(unique_ips)}")
    print(f"  Domains: {len(unique_domains)}")

if __name__ == "__main__":
    main()

Extracting IOCs from digital evidence file...

DIGITAL EVIDENCE IOC EXTRACTION REPORT
Source file: evidence_text.txt
File size: 1868 bytes

--- URLS (10 found, 10 unique) ---
ftp://files.example.com/evidence.zip
http://malicious-site.tk/payload.exe
https://amazon.com.phishing-site.ga/update
https://bit.ly/3xYzK9
https://docs.google.com/document/d/123/edit
https://github.com/user/repo
https://paypal-login.verify-account.ml/login
https://secure-bank-login.cf/verify
https://www.google.com/search?q=forensics
https://www.linkedin.com/feed/

--- EMAIL ADDRESSES (6 found, 6 unique) ---
abuse@iana.org
admin@company.com
attacker@malicious.com
noreply@github.com
security@microsoft.com
support@paypal.com

--- IP ADDRESSES (11 found, 8 unique) ---
1.1.1.1
10.0.0.100
172.16.0.50
192.168.1.10
192.168.100.50
198.51.100.23
203.0.113.45
8.8.8.8

--- DOMAIN NAMES (18 found, 18 unique) ---
amazon.com.phishing-site.ga
bit.ly
company.com
docs.google.com
example.com
files.example.com
github.com
google.com
i

<>:88: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:88: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
/tmp/ipykernel_12465/262731775.py:88: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  - HKCU\Software\Microsoft\Windows\CurrentVersion\Run


## **Result**
This the program successfully extracts URLs, email addresses, IP addresses, and domain names from a digital evidence text file.